# 21SJ-14K JAN 日次EDAノート

このノートでは、JANごとの日次売上を解析します。
売上・入荷・欠品期間を同時に確認できます。

## 使い方
1. 上から順に実行します。
2. `jan_summary` で対象JANを確認します。
3. CELL 6 だけ実行すれば、全JANを連続描画します。



In [ ]:
# セル1: ライブラリ読み込みと描画基本設定
# ノート全体で使うスタイル・フォント・基準パスを定義します


from pathlib import Path
import csv
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.style.use('ggplot')

plt.rcParams['font.family'] = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path.cwd()
print('BASE_DIR =', BASE_DIR)


In [ ]:
# セル2: 入力CSVファイルの自動検出
# 欠品データと入荷データは、ファイル名と中身パターンで判定します


all_csv = sorted(BASE_DIR.glob('*.csv'))
if not all_csv:
    raise FileNotFoundError('No CSV files found in current directory.')

jan_list_path = next((p for p in all_csv if p.name.startswith('21SJ-14K') and 'JAN' in p.name), None)
sales_path = next((p for p in all_csv if p.name.startswith('21SJ-14K') and 'JAN' not in p.name), None)

candidates_452 = [p for p in all_csv if '_452' in p.name]
stockout_path = None
receipt_path = None


def _is_jan(v: str) -> bool:
    return bool(re.fullmatch(r'452\d{10}', str(v).strip()))


def _is_date(v: str) -> bool:
    return bool(re.fullmatch(r'(\d{8}|\d{4}-\d{2}-\d{2})', str(v).strip()))


def _is_num(v: str) -> bool:
    return bool(re.fullmatch(r'-?\d+(\.\d+)?', str(v).strip()))


for p in candidates_452:
    with open(p, 'r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.reader(f))
    if len(rows) < 2:
        continue

    r1, r2 = rows[0], rows[1]
    if len(r2) >= 4:
        if (not _is_jan(r1[0])) and _is_jan(r2[0]) and _is_date(r2[1]) and _is_date(r2[2]) and _is_num(r2[3]):
            stockout_path = p
            break


for p in candidates_452:
    if p == stockout_path:
        continue
    with open(p, 'r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.reader(f))
    if not rows:
        continue

    if len(rows) >= 2 and len(rows[1]) >= 3:
        r1, r2 = rows[0], rows[1]
        if (not _is_jan(r1[0])) and _is_jan(r2[0]) and _is_date(r2[1]) and _is_num(r2[2]):
            if float(r2[2]) > 0 and '-' in str(r2[1]):
                receipt_path = p
                break

if receipt_path is None:
    for p in candidates_452:
        if p == stockout_path:
            continue
        with open(p, 'r', encoding='utf-8-sig', newline='') as f:
            rows = list(csv.reader(f))
        if not rows:
            continue

        r1 = rows[0]
        if len(r1) >= 3 and _is_jan(r1[0]) and _is_date(r1[1]) and _is_num(r1[2]) and float(r1[2]) > 0:
            receipt_path = p
            break


required = {
    'jan_list_path': jan_list_path,
    'sales_path': sales_path,
    'stockout_path': stockout_path,
    'receipt_path': receipt_path,
}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

for k, v in required.items():
    print(f'{k}: {v.name}')


In [ ]:
# セル3: データ読み込みと前処理
# daily_sales・receipts_daily・stockouts の3つの基礎テーブルを作成します



def parse_mixed_date(series: pd.Series) -> pd.Series:
    """混在形式の日付文字列をdatetimeへ変換する"""
    s = series.astype(str).str.strip().str.replace('.0', '', regex=False)
    dt = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    dt2 = pd.to_datetime(s, format='%Y%m%d', errors='coerce')
    return dt.fillna(dt2)


jan_raw = pd.read_csv(jan_list_path, header=None, usecols=[0], names=['jan'], dtype=str, encoding='utf-8-sig')
jan_raw['jan'] = jan_raw['jan'].astype(str).str.strip().str.replace('"', '', regex=False)
target_jans = sorted(set(jan_raw.loc[jan_raw['jan'].str.fullmatch(r'452\d{10}', na=False), 'jan']))


sales_cols = [
    'year', 'month', 'sales_date', 'customer_code', 'customer_name',
    'jan', 'item_name', 'model', 'color', 'size', 'region', 'sales_qty', 'industry'
]

sales = pd.read_csv(sales_path, header=None, names=sales_cols, dtype=str, encoding='utf-8-sig')
sales['jan'] = sales['jan'].astype(str).str.strip()
sales = sales[sales['jan'].isin(target_jans)].copy()
sales['sales_date'] = parse_mixed_date(sales['sales_date'])
sales['sales_qty'] = pd.to_numeric(sales['sales_qty'], errors='coerce').fillna(0)
sales = sales.dropna(subset=['sales_date'])

daily_sales = sales.groupby(['jan', 'sales_date'], as_index=False)['sales_qty'].sum()

jan_name_map = (
    sales.dropna(subset=['item_name'])
    .sort_values(['jan', 'sales_date'])
    .drop_duplicates('jan')
    .set_index('jan')['item_name']
    .to_dict()
)


stock_raw = pd.read_csv(stockout_path, dtype=str, encoding='utf-8-sig').iloc[:, :4].copy()
stock_raw.columns = ['jan', 'stockout_start', 'restock_date', 'days_out']
stock_raw['jan'] = stock_raw['jan'].astype(str).str.strip()
stock_raw = stock_raw[stock_raw['jan'].isin(target_jans)].copy()
stock_raw['stockout_start'] = parse_mixed_date(stock_raw['stockout_start'])
stock_raw['restock_date'] = parse_mixed_date(stock_raw['restock_date'])
stock_raw['days_out'] = pd.to_numeric(stock_raw['days_out'], errors='coerce')
stockouts = stock_raw


preview = pd.read_csv(receipt_path, nrows=1, header=None, dtype=str, encoding='utf-8-sig')
first_val = str(preview.iloc[0, 0]).strip() if not preview.empty else ''

if re.fullmatch(r'452\d{10}', first_val):
    rec_raw = pd.read_csv(
        receipt_path,
        header=None,
        names=['jan', 'receipt_date', 'receipt_qty'],
        dtype=str,
        encoding='utf-8-sig'
    )
else:
    rec_raw = pd.read_csv(receipt_path, dtype=str, encoding='utf-8-sig').iloc[:, :3].copy()
    rec_raw.columns = ['jan', 'receipt_date', 'receipt_qty']

rec_raw['jan'] = rec_raw['jan'].astype(str).str.strip()
rec_raw = rec_raw[rec_raw['jan'].isin(target_jans)].copy()
rec_raw['receipt_date'] = parse_mixed_date(rec_raw['receipt_date'])
rec_raw['receipt_qty'] = pd.to_numeric(rec_raw['receipt_qty'], errors='coerce').fillna(0)
rec_raw = rec_raw.dropna(subset=['receipt_date'])

receipts_daily = rec_raw.groupby(['jan', 'receipt_date'], as_index=False)['receipt_qty'].sum()


print('target_jan_count =', len(target_jans))
print('sales_unique_jan =', daily_sales['jan'].nunique())
print('sales_date_range =', daily_sales['sales_date'].min(), 'to', daily_sales['sales_date'].max())
print('stockout_unique_jan =', stockouts['jan'].nunique())
print('receipt_unique_jan =', receipts_daily['jan'].nunique())







In [ ]:
# セル4: JANサマリの作成
# 売上数量と売上日数の降順で描画順を決めます


jan_summary = (
    daily_sales.groupby('jan', as_index=False)
    .agg(
        sales_total_qty=('sales_qty', 'sum'),
        sales_days=('sales_date', 'nunique')
    )
    .sort_values(['sales_total_qty', 'sales_days'], ascending=[False, False])
)
jan_summary['has_stockout'] = jan_summary['jan'].isin(stockouts['jan'].unique())
jan_summary['has_receipt'] = jan_summary['jan'].isin(receipts_daily['jan'].unique())

print(jan_summary.head(20).to_string(index=False))


In [ ]:
# セル5: 描画関数群の定義
# 期間決定・目盛調整・欠品帯描画・JAN単位描画をまとめて定義します


GLOBAL_START = daily_sales['sales_date'].min()
GLOBAL_END = daily_sales['sales_date'].max()


def resolve_plot_range(jan_code: str, date_from=None, date_to=None, range_mode='active', pad_days=21):
    """
    Resolve chart date range.

    range_mode:
    - 'active': use this JAN's activity period (+/- pad_days)
    - 'global': use global dataset period
    """
    jan_code = str(jan_code).strip()
    js = daily_sales[daily_sales['jan'] == jan_code]
    jr = receipts_daily[receipts_daily['jan'] == jan_code]
    jz = stockouts[stockouts['jan'] == jan_code]

    if date_from is not None and date_to is not None:
        return pd.to_datetime(date_from), pd.to_datetime(date_to)

    if range_mode == 'global':
        start = GLOBAL_START if date_from is None else pd.to_datetime(date_from)
        end = GLOBAL_END if date_to is None else pd.to_datetime(date_to)
        return start, end

    bounds = []
    if not js.empty:
        bounds += [js['sales_date'].min(), js['sales_date'].max()]
    if not jr.empty:
        bounds += [jr['receipt_date'].min(), jr['receipt_date'].max()]
    if not jz.empty:
        bounds += [jz['stockout_start'].min(), jz['restock_date'].max()]

    if not bounds:
        start = GLOBAL_START if date_from is None else pd.to_datetime(date_from)
        end = GLOBAL_END if date_to is None else pd.to_datetime(date_to)
        return start, end

    start = min(bounds) if date_from is None else pd.to_datetime(date_from)
    end = max(bounds) if date_to is None else pd.to_datetime(date_to)

    if date_from is None:
        start = start - pd.Timedelta(days=pad_days)
    if date_to is None:
        end = end + pd.Timedelta(days=pad_days)

    return start, end


def set_adaptive_date_ticks(ax, start, end):
    """期間長に応じてx軸目盛りを調整する"""
    span_days = int((end - start).days) + 1

    if span_days <= 45:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    elif span_days <= 120:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    elif span_days <= 420:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    else:
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))


def build_daily_series(jan_code: str, start, end):
    """欠損日を0で埋めた日次系列を作成する"""
    jan_code = str(jan_code).strip()

    s = (
        daily_sales[daily_sales['jan'] == jan_code]
        .groupby('sales_date', as_index=True)['sales_qty']
        .sum()
    )
    r = (
        receipts_daily[receipts_daily['jan'] == jan_code]
        .groupby('receipt_date', as_index=True)['receipt_qty']
        .sum()
    )

    idx = pd.date_range(start, end, freq='D')
    y_sales = s.reindex(idx, fill_value=0).astype(float)
    y_receipt = r.reindex(idx, fill_value=0).astype(float)
    return idx, y_sales, y_receipt


def summarize_jan(jan_code: str) -> pd.Series:
    """対象JANの要約統計を返す"""
    jan_code = str(jan_code).strip()
    js = daily_sales[daily_sales['jan'] == jan_code]
    jr = receipts_daily[receipts_daily['jan'] == jan_code]
    jz = stockouts[stockouts['jan'] == jan_code]

    return pd.Series({
        'jan': jan_code,
        'item_name': jan_name_map.get(jan_code, ''),
        'sales_total_qty': float(js['sales_qty'].sum()) if not js.empty else 0.0,
        'sales_days': int(js['sales_date'].nunique()) if not js.empty else 0,
        'receipt_total_qty': float(jr['receipt_qty'].sum()) if not jr.empty else 0.0,
        'receipt_days': int(len(jr)),
        'stockout_periods': int(len(jz)),
        'stockout_total_days': float(jz['days_out'].fillna(0).sum()) if not jz.empty else 0.0,
    })


def _draw_stockout_spans(ax, stock_df, start, end, show_days_label=True):
    """欠品（売り逃し）期間の帯と日数ラベルを描画する"""
    first = True
    label_row = 0

    for _, row in stock_df.iterrows():
        s, e = row['stockout_start'], row['restock_date']
        if pd.notna(s) and pd.notna(e) and e >= s:
            s_clip = max(s, start)
            e_clip = min(e, end)
            if e_clip >= s_clip:
                ax.axvspan(s_clip, e_clip, color='tab:red', alpha=0.12, label='Stockout Period' if first else None, zorder=0)
                first = False

                if show_days_label:
                    days = row.get('days_out', np.nan)
                    if pd.isna(days):
                        days = (e - s).days
                    try:
                        days_int = int(round(float(days)))
                    except Exception:
                        days_int = None

                    if days_int is not None:
                        mid = s_clip + (e_clip - s_clip) / 2
                        y_frac = 0.96 if (label_row % 2 == 0) else 0.88
                        ax.text(
                            mid,
                            y_frac,
                            f'売り逃し {days_int}日',
                            transform=ax.get_xaxis_transform(),
                            ha='center',
                            va='top',
                            fontsize=8,
                            color='tab:red',
                            bbox={'boxstyle': 'round,pad=0.15', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.65},
                            zorder=5,
                        )
                        label_row += 1

def plot_jan_daily(
    jan_code: str,
    date_from=None,
    date_to=None,
    range_mode='active',
    pad_days=21,
    figsize=(18, 7),
    sales_line_mode='sales_days',  # 'sales_days' or 'all_days'
    show_ma7=True,
    show_yearly_cum=True,
    show_sales_date_labels=True,
    show_stockout_days=True,
):
    """JAN単位の日次グラフを描画する

    sales_line_mode:
    - 'sales_days': 売れた日のみ線を表示（0日のギザギザを抑える）
    - 'all_days'  : 0日を含めて連続線を表示
    """
    jan_code = str(jan_code).strip()
    start, end = resolve_plot_range(jan_code, date_from=date_from, date_to=date_to, range_mode=range_mode, pad_days=pad_days)

    idx, y_sales, y_receipt = build_daily_series(jan_code, start=start, end=end)
    jz = stockouts[stockouts['jan'] == jan_code].copy()
    jz = jz[(jz['restock_date'] >= start) & (jz['stockout_start'] <= end)]

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=figsize,
        sharex=True,
        constrained_layout=True,
        gridspec_kw={'height_ratios': [3.2, 1.6], 'hspace': 0.08}
    )

    _draw_stockout_spans(ax1, jz, start, end, show_days_label=show_stockout_days)
    _draw_stockout_spans(ax2, jz, start, end, show_days_label=False)

    y_plot = y_sales.copy()
    if sales_line_mode == 'sales_days':
        y_plot = y_plot.where(y_plot > 0, np.nan)

    ax1.plot(
        y_plot.index,
        y_plot.values,
        color='tab:blue',
        linewidth=1.5,
        marker='o',
        markersize=3,
        alpha=0.95,
        label='Daily Sales Qty'
    )


    nz_sales = y_sales[y_sales > 0]
    if len(nz_sales) > 0:
        ax1.scatter(nz_sales.index, nz_sales.values, color='tab:blue', s=20, alpha=0.9, zorder=4, label='Sales Days')

        if show_sales_date_labels and len(nz_sales) <= 40:
            for j, (d, q) in enumerate(nz_sales.items()):
                yoff = 8 if (j % 2 == 0) else 14
                ax1.annotate(
                    d.strftime('%m-%d'),
                    (d, q),
                    textcoords='offset points',
                    xytext=(0, yoff),
                    ha='center',
                    fontsize=7,
                    color='tab:blue',
                )

    if show_ma7:
        ma7 = y_sales.rolling(window=7, min_periods=1).mean()
        ax1.plot(ma7.index, ma7.values, color='black', linewidth=1.2, alpha=0.8, label='Sales MA(7)')

    # 年ごと累計販売数（オレンジ線）を上段に重ねる
    ax1_cum = None
    if show_yearly_cum:
        ax1_cum = ax1.twinx()
        yearly_cum = y_sales.groupby(y_sales.index.year).cumsum()

        first = True
        for y in sorted(idx.year.unique()):
            m = idx.year == y
            x_seg = idx[m]
            y_seg = yearly_cum[m]
            ax1_cum.plot(
                x_seg,
                y_seg,
                color='tab:orange',
                linewidth=1.8,
                alpha=0.9,
                label='Yearly Cumulative Sales' if first else None,
            )
            first = False

        max_cum = int(np.ceil(yearly_cum.max())) if len(yearly_cum) else 0
        ax1_cum.set_ylim(0, max(1, max_cum) + 1)
        ax1_cum.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax1_cum.set_ylabel('Yearly Cumulative Sales Qty', color='tab:orange')
        ax1_cum.tick_params(axis='y', labelcolor='tab:orange')

    max_sales = int(np.ceil(y_sales.max())) if len(y_sales) else 0
    ax1.set_ylim(0, max(1, max_sales) + 1)
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax1.set_ylabel('Sales Qty/day')
    ax1.grid(axis='y', alpha=0.25)
    ax1.grid(axis='x', alpha=0)

    nz_receipt = y_receipt[y_receipt > 0]
    if len(nz_receipt) > 0:
        ax2.plot(
            nz_receipt.index,
            nz_receipt.values,
            color='tab:green',
            linewidth=1.4,
            marker='o',
            markersize=4,
            alpha=0.9,
            label='Receipt Qty/day'
        )
        ax2.vlines(nz_receipt.index, 0, nz_receipt.values, color='tab:green', alpha=0.35, linewidth=1.1)

        for j, (d, q) in enumerate(nz_receipt.items()):
            txt = f'{int(q)}' if float(q).is_integer() else f'{q:.1f}'
            yoff = 6 if (j % 2 == 0) else 12
            ax2.annotate(txt, (d, q), textcoords='offset points', xytext=(0, yoff), ha='center', fontsize=8, color='tab:green')
    else:
        ax2.plot([], [], color='tab:green', label='Receipt Qty/day')

    max_receipt = int(np.ceil(y_receipt.max())) if len(y_receipt) else 0
    ax2.set_ylim(0, max(1, max_receipt) + 1)
    ax2.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.set_ylabel('Receipt Qty/day')
    ax2.grid(axis='y', alpha=0.25)

    item_name = jan_name_map.get(jan_code, '')
    title = f'{jan_code}' if not item_name else f'{jan_code} | {item_name}'
    title += f' | {idx.min().date()} to {idx.max().date()} (daily)'
    ax1.set_title(title)

    set_adaptive_date_ticks(ax1, idx.min(), idx.max())
    set_adaptive_date_ticks(ax2, idx.min(), idx.max())
    ax2.set_xlim(idx.min() - pd.Timedelta(days=0.5), idx.max() + pd.Timedelta(days=0.5))
    plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
    ax1.tick_params(axis='x', labelbottom=True)
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()

    if show_yearly_cum and ax1_cum is not None:
        h3, l3 = ax1_cum.get_legend_handles_labels()
    else:
        h3, l3 = [], []

    ax1.legend(h1 + h2 + h3, l1 + l2 + l3, loc='upper left', frameon=True, ncols=2)

    plt.show()
    plt.close(fig)

    print(summarize_jan(jan_code).to_string())
    return fig









In [ ]:
# セル6: 全JANを連続描画（このセルだけ実行すればOK）

# 固定期間（Noneなら自動範囲）
DATE_FROM = None
DATE_TO = None

# 表示範囲モード
# 'active': JANごとの活動期間に自動ズーム（推奨）
# 'global': 全期間を固定
RANGE_MODE = 'active'

# activeモード時の前後余白（日）
PAD_DAYS = 21

# 売上折れ線モード
# 'sales_days': 売れた日だけ線をつなぐ
# 'all_days': 0日も含めて連続線
SALES_LINE_MODE = 'sales_days'

# 売上7日移動平均の表示
ADD_MA7 = False

# 年ごと累計販売数（オレンジ線）の表示
SHOW_YEARLY_CUM = True

# 売上発生日ラベルの表示（件数が多い場合は関数内で自動抑制）
SHOW_SALES_DATE_LABELS = True

# 売り逃し期間に『何日間欠品したか』のラベルを表示
SHOW_STOCKOUT_DAYS = True

# 全JANを描画対象にする
target_jans_for_plot = jan_summary['jan'].tolist()
print('plot_count =', len(target_jans_for_plot))

for i, jan in enumerate(target_jans_for_plot, start=1):
    print(f'[{i}/{len(target_jans_for_plot)}] {jan}')
    plot_jan_daily(
        jan,
        date_from=DATE_FROM,
        date_to=DATE_TO,
        range_mode=RANGE_MODE,
        pad_days=PAD_DAYS,
        sales_line_mode=SALES_LINE_MODE,
        show_ma7=ADD_MA7,
        show_yearly_cum=SHOW_YEARLY_CUM,
        show_sales_date_labels=SHOW_SALES_DATE_LABELS,
        show_stockout_days=SHOW_STOCKOUT_DAYS,
    )



